<!-- STATUS_BLOCK_v1 -->
# 5g_foraging_effect_model.ipynb — STATUS

**Pipeline position:** Pipeline [4] of [4] — final verdict

**Purpose.**  Pre-registered statistical model: 6 sign-aligned indicators, 5-check decision rule, composite Foraging Impairment Index (FII), mixed-effects model. Produces the headline DETECTED / SUGGESTIVE / NOT-DETECTED verdict for the paper.

**Inputs:**  `data/multi_day/indicators_daily.csv` (the consolidated table from `multi_day_pipeline.ipynb`)
**Outputs:** verdict printed inline; FII + indicator effect-size table

### WORKS
- 5-check decision rule (cross-system agreement, pooled effect size, cross-cycle replication, weather-robust, drift-clean)
- Composite FII with bootstrap CI
- Mixed-effects model with system random intercept
- Bootstrap 95% CIs for rank-biserial r per indicator

### PENDING
- Replace `median_ifi_s` + `vertical_deviation` with `mean_handling_time_s` + `n_distinct_flowers` in the INDICATORS list (data is ready in `indicators_daily.csv`)
- Add `mean_dbm` as continuous covariate in mixed-effects model
- Re-run the 5-check rule with the updated indicator set

## Pipeline flow (canonical)

```
data/flight_data/<date>_system_<sys>/                  ← raw PATS-C output
       │
       ▼
[1] flower_visit_pipeline.ipynb                        slow; run when raw data changes
       └── data/multi_day/flower_visit_summary.csv

[2] multi_day_pipeline.ipynb                           always run after raw data updates
       ├── data/multi_day/daily_summary.csv
       ├── data/multi_day/per_track_indicators.csv
       └── data/multi_day/indicators_daily.csv         ← the file the model consumes

       (uses outputs of [1] + dBm + data transfer)
       │
       ▼
[3] validation.ipynb                                   sensor-integrity QC, run anytime
[3] indicator_validation.ipynb                         baseline-only QC of the 6 indicators
       │
       ▼
[4] exposure_analysis.ipynb                            figures + exploratory plots
[4] 5g_foraging_effect_model.ipynb                     pre-registered verdict, FINAL output
       │
       ▼
[5] statistical_methods.ipynb                          reading guide for [4]; not data-dependent
       │
       ▼
   paper / report
```

**Used in final report:**
- `5g_foraging_effect_model.ipynb` (verdict, mixed-effects model, composite FII)
- `exposure_analysis.ipynb` (figures)
- `validation.ipynb` (Methods)
- `indicator_validation.ipynb` (Methods)
- `statistical_methods.ipynb` (reference)

---


# 5G foraging effect model

A systematic, pre-registered analysis pipeline built to answer one question, end-to-end:

> **Does the 5G transmitter measurably impair bumblebee foraging behaviour at this site, given the n = 6 vs 6 day-cycle data we currently have?**

The answer at the end of this notebook is one of three things: **detected**, **suggestive**, or **not detected / underpowered**.  The verdict is determined by a five-check rubric defined *before* running any tests, so the bar can't be moved retrospectively.

This notebook is independent of `exposure_analysis.ipynb` — it loads the same source CSVs but rebuilds every indicator from scratch under a single sign convention (positive = "5G impairs foraging") so that everything can be pooled into one composite score.  It also incorporates the lessons from that earlier notebook:

1. **Cross-system agreement matters more than within-system p.**  We look at sys 900 and sys 939 separately *and* pooled, because consistent direction across cameras is the strongest signal we can extract at this n.
2. **Weather is a real confound.**  We run every effect twice — once raw, once after partialling out daytime-mean temperature and wind.
3. **Effect-size CIs beat p-values at small n.**  We bootstrap rank-biserial r per indicator instead of relying solely on Mann-Whitney p.
4. **The composite score buys statistical power.**  Six indicators sign-aligned and averaged into a single Foraging Impairment Index gets you ~6× the effective n of any single indicator at the cost of the strong assumption that all six measure the same construct.
5. **Cross-cycle replication is the gold standard.**  The dataset has 2 ON cycles and 2 OFF cycles.  We test each cycle separately and require the *direction* of the effect to match, since a single anomalous cycle averaged in is exactly how you fool yourself with under-powered data.


## 0. Pre-registration

**Hypothesis (H1):** 5G transmitter exposure (ON condition) is associated with reduced foraging efficiency in bumblebees, manifested as fewer trips, lower return rates, more tortuous paths, longer / more variable inter-foraging intervals, more vertical deviation at frame entry, and / or more dispersed return-heading distributions.

**Null (H0):** ON and OFF days are exchangeable for all six indicators.

**Indicator suite** (sign convention: positive = predicted under H1 = "5G impairs").  All indicators are computed per (date, system) so each cycle contributes 6 ON observations and 6 OFF observations across both cameras (n = 12 vs 12 pooled).

| # | Name | Definition | Higher value = ... |
|---|------|------------|--------------------|
| 1 | `neg_exit_count` | -1 × `n_v3` | fewer trips per day |
| 2 | `neg_re_ratio` | -1 × `re_ratio_v3` | lower return-to-exit ratio |
| 3 | `path_tortuosity` | daily median `tortuosity` | more tortuous paths |
| 4 | `median_ifi_s` | daily median inter-foraging interval | longer waits between trips |
| 5 | `ifi_cv` | `std(IFI)/mean(IFI)` per day | more erratic foraging rhythm |
| 6 | `vertical_deviation` | daily median `|z_first - hive_z|` over frame entries | more vertical confusion at FOV entry |

**Decision rule (5 binary checks):**

1. **Cross-system agreement.**  ≥ 4 / 6 indicators show the same effect direction in *both* sys 900 and sys 939.
2. **Effect size.**  ≥ 3 / 6 indicators have a pooled rank-biserial |r| > 0.3.
3. **Cross-cycle replication.**  ≥ 4 / 6 indicators agree in direction between cycle 1 and cycle 2.
4. **Weather-robustness.**  ≥ 3 / 6 indicators retain |r| > 0.3 after weather covariation (T, wind).
5. **Drift-clean.**  Day-3 OFF vs Day-1 ON is statistically indistinguishable on the composite Foraging Impairment Index (MWU p > 0.05).

**Verdict thresholds:** **5/5 = detected**, **3-4/5 = suggestive**, **0-2/5 = not detected / underpowered**.

We also run a **separate population-level test on the heading-dispersion R for frame-return headings**, since the existing analysis showed that's the cleanest single signal in the data.  This is reported alongside the composite verdict, not folded into it.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import false_discovery_control, mannwhitneyu, spearmanr

# Optional: statsmodels for mixed-effects model
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    HAS_SM = True
except ImportError:
    HAS_SM = False
    print("statsmodels not available - mixed-effects section will be skipped.")

RNG = np.random.default_rng(42)  # global seed for bootstrap / permutation
N_BOOT = 5000
N_PERM = 5000


In [ ]:
# Cycle schedule (matches exposure_analysis.ipynb)
CYCLE_ANCHOR   = pd.Timestamp("2026-04-23")
CYCLE_ON_DAYS  = 3
CYCLE_OFF_DAYS = 3
CYCLE_LEN      = CYCLE_ON_DAYS + CYCLE_OFF_DAYS

def condition_for(d):
    d = pd.Timestamp(d).normalize()
    if d < CYCLE_ANCHOR:
        return "BASELINE"
    return "ON" if (d - CYCLE_ANCHOR).days % CYCLE_LEN < CYCLE_ON_DAYS else "OFF"

def cycle_id(d):
    """Integer cycle index (0, 1, 2, ...) starting from CYCLE_ANCHOR.
    BASELINE returns NaN."""
    d = pd.Timestamp(d).normalize()
    if d < CYCLE_ANCHOR:
        return float("nan")
    return (d - CYCLE_ANCHOR).days // CYCLE_LEN

# Hive (x, y, z) per system - z is needed for the vertical_deviation indicator.
HIVE_BY_SYSTEM = {
    900: np.array([-0.04,  -0.665, -1.195]),
    939: np.array([-0.086, -0.828, -1.045]),
}


## 1. Load all data

Three CSVs from `data/multi_day/` and one KNMI hourly weather file at `../../data/wind_data_04-19_to_05-06.txt`.


In [ ]:
DATA_DIR  = Path("data/multi_day")
WIND_FILE = Path("../../data/wind_data_04-19_to_05-06.txt")

daily  = pd.read_csv(DATA_DIR / "daily_summary.csv")
tracks = pd.read_csv(DATA_DIR / "per_track_indicators.csv", parse_dates=["ts"])
geom   = pd.read_csv(DATA_DIR / "track_geometry.csv")

for df in (daily, tracks, geom):
    df["date"] = pd.to_datetime(df["date"])
    df["condition"] = df["date"].apply(condition_for)
    df["cycle"]     = df["date"].apply(cycle_id)

systems = sorted(daily["system_id"].unique())
print(f"daily : {daily.shape}")
print(f"tracks: {tracks.shape}")
print(f"geom  : {geom.shape}")
print(f"systems: {systems}")
print(f"\nday counts by condition:")
print(daily.groupby(["condition", "system_id"]).size().unstack(fill_value=0))


In [ ]:
# KNMI weather - daytime mean T and mean wind speed per date.
if WIND_FILE.exists():
    KNMI_COLS = ["STN","YYYYMMDD","HH","DD","FH","FF","FX","T","T10N","TD",
                 "SQ","Q","DR","RH","P","VV","N","U","WW","IX","M","R","S","O","Y"]
    raw = pd.read_csv(WIND_FILE, comment="#", names=KNMI_COLS,
                      skipinitialspace=True, na_values=["", " "])
    raw["ts_utc"]   = (pd.to_datetime(raw["YYYYMMDD"].astype(int), format="%Y%m%d")
                       + pd.to_timedelta(raw["HH"].astype(int) - 1, unit="h"))
    raw["ts_utc"]   = raw["ts_utc"].dt.tz_localize("UTC")
    raw["ts_local"] = raw["ts_utc"].dt.tz_convert("Europe/Amsterdam")
    raw["date"]     = raw["ts_local"].dt.date
    raw["hour"]     = raw["ts_local"].dt.hour
    raw["T_C"]      = raw["T"]  / 10.0
    raw["wind_ms"]  = raw["FH"] / 10.0
    daytime = raw[raw["hour"].between(8, 17)]
    weather_daily = (daytime.groupby("date")
                              .agg(T_mean_day=("T_C",     "mean"),
                                   wind_mean_day=("wind_ms", "mean"))
                              .reset_index())
    weather_daily["date"] = pd.to_datetime(weather_daily["date"])
    print(f"Weather: {len(weather_daily)} dates, "
          f"T {weather_daily['T_mean_day'].min():.1f}-{weather_daily['T_mean_day'].max():.1f} degC, "
          f"wind {weather_daily['wind_mean_day'].min():.1f}-{weather_daily['wind_mean_day'].max():.1f} m/s")
else:
    weather_daily = None
    print(f"WARNING: {WIND_FILE.resolve()} not found - weather-robustness check will be skipped.")


## 2. Build the indicator table (one row per date × system)

Six per-(date, system) values, all sign-aligned so a *positive* value supports H1.


In [ ]:
# IFI: time between consecutive hive_exit_v3 within (system, date)
exits = tracks[(tracks["hive_exit_v3"] == True) & tracks["ts"].notna()].copy()
exits = exits.sort_values(["system_id", "date", "ts"]).reset_index(drop=True)
exits["ifi_s"] = exits.groupby(["system_id", "date"])["ts"].diff().dt.total_seconds()

ifi_per_day = (exits.dropna(subset=["ifi_s"])
                     .groupby(["date", "system_id"])["ifi_s"]
                     .agg(median_ifi_s="median", mean="mean", std="std", count="count")
                     .reset_index())
ifi_per_day["ifi_cv"] = np.where(
    (ifi_per_day["mean"] > 0) & (ifi_per_day["count"] >= 2),
    ifi_per_day["std"] / ifi_per_day["mean"], np.nan)

# path_tortuosity: median per day
tort_daily = (tracks.dropna(subset=["tortuosity"])
                    .groupby(["date", "system_id"])["tortuosity"]
                    .median()
                    .reset_index(name="path_tortuosity"))

# vertical_deviation: median |z_first - hive_z| over frame entries (entry_at_hive == False)
frame_entries = geom[~geom["entry_at_hive"]].copy()
frame_entries["hive_z"] = frame_entries["system_id"].map({s: HIVE_BY_SYSTEM[s][2] for s in systems})
frame_entries["z_dev"]  = (frame_entries["z_first"] - frame_entries["hive_z"]).abs()
vert_daily = (frame_entries.dropna(subset=["z_dev"])
                            .groupby(["date", "system_id"])["z_dev"]
                            .median()
                            .reset_index(name="vertical_deviation"))

# Assemble
ind = (daily[["date", "system_id", "n_v3", "re_ratio_v3", "condition", "cycle"]]
       .rename(columns={"n_v3": "exit_count", "re_ratio_v3": "re_ratio"})
       .merge(tort_daily, on=["date", "system_id"], how="left")
       .merge(ifi_per_day[["date", "system_id", "median_ifi_s", "ifi_cv"]],
              on=["date", "system_id"], how="left")
       .merge(vert_daily, on=["date", "system_id"], how="left"))

# Sign-align: positive = predicted under H1 (5G impairs)
ind["neg_exit_count"] = -ind["exit_count"]
ind["neg_re_ratio"]   = -ind["re_ratio"]
# path_tortuosity, median_ifi_s, ifi_cv, vertical_deviation: already positive-impairment

# Add weather
if weather_daily is not None:
    ind = ind.merge(weather_daily, on="date", how="left")

INDICATORS = ["neg_exit_count", "neg_re_ratio", "path_tortuosity",
              "median_ifi_s", "ifi_cv", "vertical_deviation"]

# Restrict to ON / OFF
ind_on_off = ind[ind["condition"].isin(["ON", "OFF"])].copy()
print(f"ON/OFF day-system rows: {len(ind_on_off)}")
print(f"  ON  per system: {ind_on_off[ind_on_off['condition']=='ON'].groupby('system_id').size().to_dict()}")
print(f"  OFF per system: {ind_on_off[ind_on_off['condition']=='OFF'].groupby('system_id').size().to_dict()}")
print(f"\nMissing values per indicator:")
print(ind_on_off[INDICATORS].isna().sum())


## 3. Pre-flight diagnostics

Two checks that *must* pass before the main analysis is interpretable.


### 3.1 Drift check (Day-3 OFF vs Day-1 ON)

If bees are already drifting before exposure starts, ON-vs-OFF differences are seasonal artefacts. We compare the last day of each OFF block to the first day of the immediately following ON block — these days are 24 hours apart, so seasonal drift has barely had time to act.


In [ ]:
def block_pos(d, on=True):
    d = pd.Timestamp(d).normalize()
    if d < CYCLE_ANCHOR: return float("nan")
    pos = (d - CYCLE_ANCHOR).days % CYCLE_LEN
    if on:
        return float(pos + 1) if pos < CYCLE_ON_DAYS else float("nan")
    else:
        return float(pos - CYCLE_ON_DAYS + 1) if pos >= CYCLE_ON_DAYS else float("nan")

drift_check = ind.copy()
drift_check["on_day"]  = drift_check["date"].apply(lambda d: block_pos(d, on=True))
drift_check["off_day"] = drift_check["date"].apply(lambda d: block_pos(d, on=False))
d3_off = drift_check[drift_check["off_day"] == 3.0].assign(group="D3_OFF")
d1_on  = drift_check[drift_check["on_day"]  == 1.0].assign(group="D1_ON")
drift_pair = pd.concat([d3_off, d1_on], ignore_index=True)

print(f"{'indicator':22s} {'sys':>4s} {'D3_OFF':>9s} {'D1_ON':>9s} {'p (MWU)':>9s}")
clean = total = 0
for ind_name in INDICATORS:
    for sid in systems:
        sub = drift_pair[drift_pair["system_id"] == sid]
        a = sub.loc[sub["group"] == "D3_OFF", ind_name].dropna()
        b = sub.loc[sub["group"] == "D1_ON",  ind_name].dropna()
        if len(a) and len(b):
            try:
                p = mannwhitneyu(a, b, alternative="two-sided").pvalue
            except ValueError:
                p = float("nan")
            total += 1
            if not np.isnan(p) and p > 0.05: clean += 1
        else:
            p = float("nan")
        print(f"  {ind_name:20s} {sid:>4d} {a.median():>9.3f} {b.median():>9.3f} {p:>9.3f}")
DRIFT_CLEAN = clean == total and total > 0
print(f"\nDrift-clean pairs: {clean}/{total}  -> Check 5 = {DRIFT_CLEAN}")


### 3.2 Weather balance check

If ON-days and OFF-days had systematically different weather, the ON/OFF contrast is partially a weather contrast.


In [ ]:
if weather_daily is None:
    print("Skipping weather balance check.")
    WEATHER_BALANCED = None
else:
    bal = ind_on_off.dropna(subset=["T_mean_day", "wind_mean_day"]).drop_duplicates(subset=["date"])
    on  = bal[bal["condition"] == "ON"]
    off = bal[bal["condition"] == "OFF"]
    print(f"{'var':>14s} {'ON med':>8s} {'OFF med':>9s} {'p (MWU)':>9s}  flag")
    flagged = 0
    for v in ["T_mean_day", "wind_mean_day"]:
        a, b = on[v].dropna(), off[v].dropna()
        p = mannwhitneyu(a, b, alternative="two-sided").pvalue if len(a) and len(b) else float("nan")
        flag = "  *** unbalanced (p<0.05)" if p < 0.05 else ""
        if p < 0.05: flagged += 1
        print(f"  {v:>12s} {a.median():>8.2f} {b.median():>9.2f} {p:>9.3f}{flag}")
    WEATHER_BALANCED = flagged == 0
    print(f"\nWeather-balanced: {WEATHER_BALANCED}")
    if not WEATHER_BALANCED:
        print("Weather differs between ON and OFF -- weather-robustness check (section 6) becomes critical.")


## 4. Per-indicator effect-size estimation

For each indicator we report:

- Mann-Whitney U statistic and raw p
- Rank-biserial correlation r (signed: positive = ON > OFF as predicted by H1)
- Bootstrap 95% CI on r (5 000 resamples, fixed seed)

We do this **per system** (to look for cross-system agreement) and **pooled** (n = 12 vs 12).


In [ ]:
def rank_biserial(on_vals, off_vals):
    """Rank-biserial correlation r in [-1, +1].
    +1 = every ON value > every OFF value; -1 = converse.  Sign convention here:
    positive r means ON has higher values (which, given our sign-aligned indicators,
    means ON shows MORE impairment - i.e. supports H1)."""
    on_vals  = np.asarray(on_vals,  dtype=float)
    off_vals = np.asarray(off_vals, dtype=float)
    on_vals  = on_vals[~np.isnan(on_vals)]
    off_vals = off_vals[~np.isnan(off_vals)]
    n1, n2 = len(on_vals), len(off_vals)
    if n1 == 0 or n2 == 0: return float("nan")
    U_on = mannwhitneyu(on_vals, off_vals, alternative="two-sided").statistic
    return float(2 * U_on / (n1 * n2) - 1)

def bootstrap_r_ci(on_vals, off_vals, n_boot=N_BOOT, alpha=0.05, rng=None):
    rng = rng or np.random.default_rng(42)
    on_vals  = np.asarray(on_vals,  dtype=float)
    off_vals = np.asarray(off_vals, dtype=float)
    on_vals  = on_vals[~np.isnan(on_vals)]
    off_vals = off_vals[~np.isnan(off_vals)]
    if len(on_vals) < 2 or len(off_vals) < 2: return (float("nan"), float("nan"))
    rs = np.empty(n_boot)
    for i in range(n_boot):
        a = rng.choice(on_vals,  size=len(on_vals),  replace=True)
        b = rng.choice(off_vals, size=len(off_vals), replace=True)
        rs[i] = rank_biserial(a, b)
    lo, hi = np.percentile(rs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)

def per_indicator_test(df, ind_name, group_col="system_id"):
    rows = []
    for grp_val, sub in df.groupby(group_col):
        on  = sub.loc[sub["condition"] == "ON",  ind_name].dropna()
        off = sub.loc[sub["condition"] == "OFF", ind_name].dropna()
        if len(on) and len(off):
            U, p = mannwhitneyu(on, off, alternative="two-sided")
            r    = rank_biserial(on, off)
            lo, hi = bootstrap_r_ci(on, off, rng=np.random.default_rng(int(grp_val) + hash(ind_name) % 10000))
        else:
            U, p, r, lo, hi = float("nan"), 1.0, float("nan"), float("nan"), float("nan")
        rows.append({group_col: grp_val, "indicator": ind_name,
                     "n_on": len(on), "n_off": len(off),
                     "U": U, "p_raw": p, "r": r, "r_lo": lo, "r_hi": hi})
    return rows

# Per-system table
per_sys = [r for ind_name in INDICATORS
             for r in per_indicator_test(ind_on_off, ind_name, "system_id")]
per_sys_df = pd.DataFrame(per_sys)

# Pooled (use a dummy group)
ind_on_off_pool = ind_on_off.assign(_pool=0)
per_pool = [r for ind_name in INDICATORS
              for r in per_indicator_test(ind_on_off_pool, ind_name, "_pool")]
per_pool_df = pd.DataFrame(per_pool).rename(columns={"_pool": "scope"})
per_pool_df["scope"] = "pooled"

# BH adjust p across the 6 indicators within each system + pooled
for sid in list(systems) + ["pooled"]:
    if sid == "pooled":
        mask = per_pool_df["indicator"].isin(INDICATORS)
        per_pool_df.loc[mask, "p_adj_BH"] = false_discovery_control(per_pool_df.loc[mask, "p_raw"].values)
    else:
        mask = per_sys_df["system_id"] == sid
        per_sys_df.loc[mask, "p_adj_BH"] = false_discovery_control(per_sys_df.loc[mask, "p_raw"].values)

print("=" * 90)
print("Per-system effect sizes (rank-biserial r, 95% bootstrap CI)")
print("=" * 90)
print(f"{'sys':>4s} {'indicator':22s} {'n_on/off':>10s} {'r':>7s} {'95% CI':>17s} {'p':>7s} {'p_BH':>7s}  flag")
for _, row in per_sys_df.iterrows():
    flag = ""
    if abs(row["r"]) > 0.3:                       flag += "|r|>0.3 "
    if row["r_lo"] > 0:                            flag += "CI>0 "
    if row["p_adj_BH"] < 0.05:                     flag += "BH<.05 "
    print(f"  {int(row['system_id']):>2d} {row['indicator']:22s} "
          f"{int(row['n_on']):>3d}/{int(row['n_off']):<3d}  "
          f"{row['r']:>+7.3f} [{row['r_lo']:>+5.2f},{row['r_hi']:>+5.2f}] "
          f"{row['p_raw']:>7.3f} {row['p_adj_BH']:>7.3f}  {flag}")

print()
print("=" * 90)
print("Pooled across systems (n=12 vs n=12)")
print("=" * 90)
print(f"{'indicator':22s} {'r':>7s} {'95% CI':>17s} {'p':>7s} {'p_BH':>7s}  flag")
for _, row in per_pool_df.iterrows():
    flag = ""
    if abs(row["r"]) > 0.3:                       flag += "|r|>0.3 "
    if row["r_lo"] > 0:                            flag += "CI>0 "
    if row["p_adj_BH"] < 0.05:                     flag += "BH<.05 "
    print(f"  {row['indicator']:22s} {row['r']:>+7.3f} "
          f"[{row['r_lo']:>+5.2f},{row['r_hi']:>+5.2f}] "
          f"{row['p_raw']:>7.3f} {row['p_adj_BH']:>7.3f}  {flag}")


## 5. Cross-system agreement (Check 1)

For each indicator, do sys 900 and sys 939 agree on the sign of the effect?


In [ ]:
agree_rows = []
for ind_name in INDICATORS:
    rs = per_sys_df[per_sys_df["indicator"] == ind_name].set_index("system_id")["r"]
    if len(rs) >= 2:
        signs = np.sign(rs.values)
        agree = bool((signs[0] == signs[1]) and (signs[0] != 0))
    else:
        agree = False
    agree_rows.append({"indicator": ind_name,
                       "r_sys900": rs.get(900, float("nan")),
                       "r_sys939": rs.get(939, float("nan")),
                       "agreement": agree})
agree_df = pd.DataFrame(agree_rows)
print(f"{'indicator':22s} {'r sys900':>9s} {'r sys939':>9s}  agreement")
for _, row in agree_df.iterrows():
    print(f"  {row['indicator']:22s} {row['r_sys900']:>+9.3f} {row['r_sys939']:>+9.3f}  "
          f"{'AGREE' if row['agreement'] else 'disagree'}")
n_agree = int(agree_df["agreement"].sum())
CHECK_1 = n_agree >= 4
print(f"\nCross-system agreement: {n_agree}/{len(agree_df)} indicators")
print(f"Check 1 ({n_agree} >= 4): {CHECK_1}")


## 6. Effect size in the pooled analysis (Check 2)

How many indicators have pooled |r| > 0.3 — i.e. show a moderate-or-larger effect once the cameras are pooled?


In [ ]:
big_pooled = (per_pool_df["r"].abs() > 0.3).sum()
CHECK_2 = big_pooled >= 3
print(f"Pooled indicators with |r| > 0.3:")
for _, row in per_pool_df.iterrows():
    flag = " *" if abs(row["r"]) > 0.3 else ""
    print(f"  {row['indicator']:22s}  r = {row['r']:+.3f}{flag}")
print(f"\nTotal: {big_pooled}/6")
print(f"Check 2 ({big_pooled} >= 3): {CHECK_2}")


## 7. Cross-cycle replication (Check 3)

The dataset has 2 ON cycles (cycle 0: Apr 23-25, cycle 1: Apr 29-May 1) and 2 OFF cycles (cycle 0: Apr 26-28, cycle 1: May 2-4).  We compare cycle-0 ON vs cycle-0 OFF *separately from* cycle-1 ON vs cycle-1 OFF.  An indicator that flips direction between cycles is unreliable; an indicator that stays in the same direction in both cycles is reproducible.


In [ ]:
cycle_pairs = []
# Cycle 0 = ON cycle 0 (Apr 23-25) + OFF cycle 0 (Apr 26-28)
# Cycle 1 = ON cycle 1 (Apr 29-May 1) + OFF cycle 1 (May 2-4)
# In our cycle_id scheme, an OFF block sits in the same cycle index as the ON block before it.
for cyc in [0, 1]:
    sub = ind_on_off[ind_on_off["cycle"] == cyc]
    for ind_name in INDICATORS:
        on  = sub.loc[sub["condition"] == "ON",  ind_name].dropna()
        off = sub.loc[sub["condition"] == "OFF", ind_name].dropna()
        if len(on) and len(off):
            r = rank_biserial(on, off)
        else:
            r = float("nan")
        cycle_pairs.append({"cycle": cyc, "indicator": ind_name, "r": r,
                            "n_on": len(on), "n_off": len(off)})
cycle_df = pd.DataFrame(cycle_pairs)

print(f"{'indicator':22s} {'r cyc0':>8s} {'r cyc1':>8s}  replicated?")
replicated = 0
for ind_name in INDICATORS:
    r0 = cycle_df.loc[(cycle_df["cycle"] == 0) & (cycle_df["indicator"] == ind_name), "r"]
    r1 = cycle_df.loc[(cycle_df["cycle"] == 1) & (cycle_df["indicator"] == ind_name), "r"]
    r0v = float(r0.iloc[0]) if len(r0) else float("nan")
    r1v = float(r1.iloc[0]) if len(r1) else float("nan")
    same = (not np.isnan(r0v)) and (not np.isnan(r1v)) and np.sign(r0v) == np.sign(r1v) and r0v != 0
    if same: replicated += 1
    print(f"  {ind_name:22s} {r0v:>+8.3f} {r1v:>+8.3f}  {'REPLICATED' if same else 'flips'}")
CHECK_3 = replicated >= 4
print(f"\nCross-cycle replication: {replicated}/6")
print(f"Check 3 ({replicated} >= 4): {CHECK_3}")


## 8. Weather robustness (Check 4)

For each indicator we fit `Y ~ T_mean_day + wind_mean_day + condition_ON` via OLS, then re-estimate the effect of condition with weather partialled out.  We approximate a weather-controlled rank-biserial r by computing it on the *residuals* (Y minus the weather-only fit).


In [ ]:
def weather_residualise(df, ind_name):
    """Return residuals of Y ~ T + wind (weather-only OLS, no condition)."""
    sub = df.dropna(subset=[ind_name, "T_mean_day", "wind_mean_day"]).copy()
    if len(sub) < 5:
        return None
    X = sub[["T_mean_day", "wind_mean_day"]].values
    X = np.column_stack([np.ones(len(X)), X])
    y = sub[ind_name].values
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    sub["_resid"] = y - X @ beta
    return sub

if weather_daily is None:
    print("No weather data - skipping Check 4.")
    CHECK_4 = None
    weather_robust_df = None
else:
    rows = []
    for ind_name in INDICATORS:
        sub = weather_residualise(ind_on_off, ind_name)
        if sub is None:
            rows.append({"indicator": ind_name, "r_raw": float("nan"),
                         "r_resid": float("nan")})
            continue
        on_resid  = sub.loc[sub["condition"] == "ON",  "_resid"].dropna()
        off_resid = sub.loc[sub["condition"] == "OFF", "_resid"].dropna()
        on_raw    = sub.loc[sub["condition"] == "ON",  ind_name].dropna()
        off_raw   = sub.loc[sub["condition"] == "OFF", ind_name].dropna()
        rows.append({"indicator": ind_name,
                     "r_raw":   rank_biserial(on_raw,   off_raw),
                     "r_resid": rank_biserial(on_resid, off_resid)})
    weather_robust_df = pd.DataFrame(rows)
    print(f"{'indicator':22s} {'r raw':>8s} {'r weather-resid':>16s}  retained?")
    retained = 0
    for _, row in weather_robust_df.iterrows():
        kept = (not np.isnan(row["r_resid"])) and abs(row["r_resid"]) > 0.3
        if kept: retained += 1
        print(f"  {row['indicator']:22s} {row['r_raw']:>+8.3f} {row['r_resid']:>+16.3f}  "
              f"{'YES' if kept else 'lost'}")
    CHECK_4 = retained >= 3
    print(f"\nWeather-robust indicators (|r_resid| > 0.3): {retained}/6")
    print(f"Check 4 ({retained} >= 3): {CHECK_4}")


## 9. Composite Foraging Impairment Index (FII)

We z-score each indicator (per system, across all ON+OFF days), then average across the 6 sign-aligned indicators per (date, system).  Higher FII = more impairment-aligned across the whole indicator suite.

This is a single test combining all the evidence into one number.  Even if no single indicator clears BH, the composite can — because averaging 6 noisy-but-directionally-consistent indicators kills variance.


In [ ]:
zind = ind_on_off.copy()
for ind_name in INDICATORS:
    # z-score within each system (so different systems' baselines don't matter)
    g = zind.groupby("system_id")[ind_name]
    zind[ind_name + "_z"] = (zind[ind_name] - g.transform("mean")) / g.transform("std")

# FII = average z-score across the 6 sign-aligned indicators per row
zind["FII"] = zind[[c + "_z" for c in INDICATORS]].mean(axis=1, skipna=True)
fii_table = zind[["date", "system_id", "cycle", "condition", "FII"]].dropna(subset=["FII"])

# Plot
fig, axes = plt.subplots(1, len(systems) + 1, figsize=(5 * (len(systems)+1), 3.8),
                          sharey=True, squeeze=False)
ORDER  = ["OFF", "ON"]
COLORS = {"OFF": "tab:green", "ON": "tab:red"}
groups = [(sid, fii_table[fii_table["system_id"] == sid], f"sys {sid}") for sid in systems]
groups.append(("pooled", fii_table, "pooled (n=12 vs 12)"))
for ax, (label, sub, title) in zip(axes[0], groups):
    data = [sub.loc[sub["condition"] == c, "FII"].dropna() for c in ORDER]
    bp = ax.boxplot(data, tick_labels=ORDER, patch_artist=True, widths=0.55)
    for patch, c in zip(bp["boxes"], ORDER):
        patch.set_facecolor(COLORS[c]); patch.set_alpha(0.7)
    for i, dvals in enumerate(data):
        ax.scatter(np.full(len(dvals), i + 1) + np.random.uniform(-0.06, 0.06, len(dvals)),
                   dvals, color="black", s=22, alpha=0.8, zorder=3)
    ax.set_title(title)
    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.set_ylabel("Foraging Impairment Index (z-score, +ve = ON-impaired)")
fig.suptitle("Composite Foraging Impairment Index (FII) - ON vs OFF", y=1.02)
plt.tight_layout(); plt.show()

# MWU + bootstrap CI on the pooled FII
on_fii  = fii_table.loc[fii_table["condition"] == "ON",  "FII"].dropna()
off_fii = fii_table.loc[fii_table["condition"] == "OFF", "FII"].dropna()
U, p_fii = mannwhitneyu(on_fii, off_fii, alternative="two-sided")
r_fii    = rank_biserial(on_fii, off_fii)
ci_fii   = bootstrap_r_ci(on_fii, off_fii, rng=np.random.default_rng(123))
print(f"\nPooled FII (n_on={len(on_fii)}, n_off={len(off_fii)}):")
print(f"  ON med  = {on_fii.median():+.3f}")
print(f"  OFF med = {off_fii.median():+.3f}")
print(f"  rank-biserial r = {r_fii:+.3f}  (95% CI {ci_fii[0]:+.2f}, {ci_fii[1]:+.2f})")
print(f"  MWU U = {U:.0f}, p = {p_fii:.4f}")
print(f"  Direction: {'ON > OFF (impairment-aligned)' if r_fii > 0 else 'ON < OFF (against H1)'}")


## 10. Mixed-effects model (statsmodels)

Pool across systems with `system_id` as a random effect.  The fixed-effect coefficient on `condition_ON` is the average effect of the transmitter on FII after accounting for system-level baseline differences.  This trades off some interpretability for more statistical power than the per-system MWU.

Skipped if `statsmodels` isn't installed.


In [ ]:
if HAS_SM:
    fii_table = fii_table.copy()
    fii_table["condition_ON"] = (fii_table["condition"] == "ON").astype(int)
    md_ = smf.mixedlm("FII ~ condition_ON", data=fii_table, groups=fii_table["system_id"])
    res = md_.fit(method="lbfgs", reml=False)
    print(res.summary().tables[1])
    coef_on = res.params.get("condition_ON", float("nan"))
    p_on    = res.pvalues.get("condition_ON", float("nan"))
    print(f"\nFII condition_ON coefficient: {coef_on:+.3f}  (p = {p_on:.4f})")
else:
    print("statsmodels not installed - skipping mixed-effects model.")


## 11. Drift-clean check on the composite (Check 5)

Same as 3.1 but tested directly on the FII rather than per-indicator.  This is the cleanest version of the drift check because the composite is what we're using in the verdict.


In [ ]:
zfull = ind.copy()
for ind_name in INDICATORS:
    g = zfull.groupby("system_id")[ind_name]
    zfull[ind_name + "_z"] = (zfull[ind_name] - g.transform("mean")) / g.transform("std")
zfull["FII"] = zfull[[c + "_z" for c in INDICATORS]].mean(axis=1, skipna=True)
zfull["on_day"]  = zfull["date"].apply(lambda d: block_pos(d, on=True))
zfull["off_day"] = zfull["date"].apply(lambda d: block_pos(d, on=False))

a = zfull.loc[zfull["off_day"] == 3.0, "FII"].dropna()
b = zfull.loc[zfull["on_day"]  == 1.0, "FII"].dropna()
if len(a) and len(b):
    p_drift = mannwhitneyu(a, b, alternative="two-sided").pvalue
else:
    p_drift = float("nan")
print(f"Day-3 OFF FII med: {a.median():+.3f}  (n={len(a)})")
print(f"Day-1 ON  FII med: {b.median():+.3f}  (n={len(b)})")
print(f"MWU p = {p_drift:.4f}")
CHECK_5 = (not np.isnan(p_drift)) and p_drift > 0.05
print(f"Check 5 (FII drift-clean, p > 0.05): {CHECK_5}")


## 12. Supplementary: heading dispersion (population-level)

Reported alongside but not folded into the verdict.  The hypothesis is that returning bees come from more dispersed directions under ON if 5G interferes with their navigation.

We use a permutation test on |R_OFF − R_ON| since scipy doesn't ship Watson's U².


In [ ]:
def R_of(deg_array):
    deg_array = np.asarray(deg_array, dtype=float)
    deg_array = deg_array[~np.isnan(deg_array)]
    if len(deg_array) == 0: return float("nan"), 0
    rad = np.deg2rad(deg_array)
    return float(np.abs(np.mean(np.exp(1j * rad)))), len(deg_array)

def perm_test_R_diff(on_deg, off_deg, n_perm=N_PERM, seed=0):
    on_deg  = np.asarray(on_deg,  dtype=float)
    off_deg = np.asarray(off_deg, dtype=float)
    on_deg  = on_deg[~np.isnan(on_deg)]
    off_deg = off_deg[~np.isnan(off_deg)]
    if len(on_deg) == 0 or len(off_deg) == 0:
        return float("nan"), float("nan"), float("nan"), float("nan")
    R_on,  _ = R_of(on_deg)
    R_off, _ = R_of(off_deg)
    obs = abs(R_on - R_off)
    pool = np.concatenate([on_deg, off_deg])
    rng = np.random.default_rng(seed)
    n1 = len(on_deg)
    count = 0
    for _ in range(n_perm):
        rng.shuffle(pool)
        a, b = pool[:n1], pool[n1:]
        ra = np.abs(np.mean(np.exp(1j * np.deg2rad(a))))
        rb = np.abs(np.mean(np.exp(1j * np.deg2rad(b))))
        if abs(ra - rb) >= obs: count += 1
    return R_on, R_off, obs, (count + 1) / (n_perm + 1)

ret_h = geom[(geom["entry_at_hive"] == False) & (geom["exit_at_hive"] == True)]
print(f"{'sys':>4s} {'R_OFF':>7s} {'R_ON':>7s} {'|dR|':>7s} {'p (perm)':>10s} {'n_off':>6s} {'n_on':>5s}  flag")
ret_significant = 0
ret_total = 0
for sid in systems:
    sub = ret_h[ret_h["system_id"] == sid]
    on  = sub.loc[sub["condition"] == "ON",  "heading_entry_deg"].dropna().values
    off = sub.loc[sub["condition"] == "OFF", "heading_entry_deg"].dropna().values
    R_on, R_off, dR, p = perm_test_R_diff(on, off, seed=int(sid))
    flag = ""
    if p < 0.05 and R_on < R_off: flag = "  *** ON more dispersed (p<.05)"
    elif p < 0.05:                flag = f"  p<.05 but R_ON > R_OFF"
    if not np.isnan(p): ret_total += 1
    if p < 0.05 and R_on < R_off: ret_significant += 1
    print(f"  {sid:>2d}   {R_off:>7.3f} {R_on:>7.3f} {dR:>7.3f} {p:>10.4f} {len(off):>6d} {len(on):>5d}{flag}")
HEADING_DISPERSION_BOTH_SYSTEMS = ret_significant == ret_total and ret_total > 0
print(f"\nBoth systems significantly more dispersed under ON: {HEADING_DISPERSION_BOTH_SYSTEMS}")


## 13. Verdict

Aggregate the five pre-registered checks into a single verdict.


In [ ]:
checks = {
    "1. Cross-system agreement (>=4/6)":          CHECK_1,
    "2. Pooled effect size (>=3/6 with |r|>0.3)": CHECK_2,
    "3. Cross-cycle replication (>=4/6)":         CHECK_3,
    "4. Weather-robust (>=3/6 retain |r|>0.3)":   CHECK_4 if CHECK_4 is not None else "skipped",
    "5. FII drift-clean (D3OFF~D1ON, p>0.05)":    CHECK_5,
}
print("=" * 70)
print("Pre-registered decision rule")
print("=" * 70)
for name, val in checks.items():
    if val is None or isinstance(val, str):
        tag = '----'
    else:
        tag = 'PASS' if bool(val) else 'fail'
    print(f"  [{tag:>4s}]  {name}")

def _is_pass(v):
    return v is not None and not isinstance(v, str) and bool(v)

def _is_evaluated(v):
    return v is not None and not isinstance(v, str)

n_pass = sum(1 for v in checks.values() if _is_pass(v))
n_eval = sum(1 for v in checks.values() if _is_evaluated(v))

if n_pass == 5:
    verdict = "DETECTED -- strong directional signal, replicated and weather-robust"
elif n_pass >= 3:
    verdict = "SUGGESTIVE -- some evidence of an effect, not all checks passed"
elif n_pass >= 1:
    verdict = "WEAK / underpowered -- limited evidence, dataset cannot distinguish"
else:
    verdict = "NOT DETECTED -- no consistent evidence of a 5G effect"

print()
print("=" * 70)
print(f"Score: {n_pass}/{n_eval} checks passed")
print(f"VERDICT: {verdict}")
print("=" * 70)
print()
print("Supplementary signal (not in verdict):")
print(f"  Heading dispersion (frame-return) more dispersed under ON in BOTH systems: "
      f"{HEADING_DISPERSION_BOTH_SYSTEMS}")
print()
print("Pooled FII summary (the headline number):")
print(f"  Median FII  ON: {on_fii.median():+.3f}")
print(f"  Median FII OFF: {off_fii.median():+.3f}")
print(f"  rank-biserial r = {r_fii:+.3f}  (95% CI {ci_fii[0]:+.2f}, {ci_fii[1]:+.2f})  "
      f"MWU p = {p_fii:.4f}")


## 14. How to extend this when more data arrives

The pipeline is designed so that adding more cycles requires no code changes — the cycle / condition functions automatically tag new dates, and the indicator table grows by 2 rows per (date, system).  Three things become possible that aren't possible at this n:

- **BH correction has room to breathe.**  At 12 vs 12, the smallest possible MWU p ≈ 3×10⁻⁵, so each individual indicator can clear BH at q = 0.05 without needing the composite trick.  At 24 vs 24 you start being able to detect smaller effect sizes too.
- **Section M (accumulation) becomes interpretable.**  At 4 cycles, Day-1 ON vs Day-3 ON has n = 4 vs 4, smallest MWU p ≈ 0.029 — i.e. capable of yielding significance, not just direction.
- **A dose-response analysis with the Grafana power data becomes feasible.**  The hourly Grafana stream is fixed-width regardless of cycle count, but more total ON-hours means more dynamic range in dBm and a better-conditioned regression.

If you want to go further than this notebook in modelling sophistication:

- Replace MWU with **mixed-effects logistic-regression on per-track success** (return / no return).  Each track is one observation, you have ~10000 of them, and you can include track-level covariates (departure time, track length, tortuosity) along with condition.
- Use a **changepoint detector on the hourly time series** to ask not "is the daily mean different" but "is there a structural break in the foraging rhythm at the moment ON starts."  More sensitive to short-lived effects.
- **Bayesian model with explicit priors on indicator effects** lets you propagate the n = 12 vs 12 uncertainty all the way to the final verdict, rather than collapsing each step to a binary check.
